## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [1]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np


SEED = 43  # random seed for reproducability (change seed, if desired)
SET_ORIGINAL_INDICES = False  # if set to true, the original paper indices are selected

np.random.seed(SEED)

In [2]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [3]:
display(business_covariates.dtypes)
display(reviews.dtypes)

display(business_covariates.head(2)["Checkin"])
display(reviews.head(2))
len(reviews["user_id"].unique())

business_id                 object
name                        object
neighborhood               float64
address                     object
city                        object
state                       object
postal_code                float64
latitude                   float64
longitude                  float64
stars                      float64
review_count                 int64
is_open                      int64
categories                  object
Checkin                      int64
chain                        int64
density                      int64
TRAIN                        int64
category                    object
FT                            bool
Price.Level                float64
Restaurant.Size            float64
Number.of.Seats            float64
ZRI                        float64
Distance.To.City.Centre    float64
dtype: object

review_id          object
business_id        object
user_id            object
date               object
stars               int64
text               object
funny               int64
cool                int64
useful              int64
Year                int64
first_year          int64
.groups            object
n_reviews           int64
AggRat            float64
language           object
tempcontiguity      int64
sentimenttext     float64
wordCount           int64
dtype: object

0    171
1     59
Name: Checkin, dtype: int64

,review_id,business_id,user_id,date,stars,text,funny,cool,useful,Year,first_year,.groups,n_reviews,AggRat,language,tempcontiguity,sentimenttext,wordCount
0,EQF-SyHb_Yg0HC9E9BppOg,--g-a85VwrdZJNf0R95GcQ,SvsoiaCf0WG7UIDJOxJ7Yg,2013-11-14,5,super fresh food..great prices. ala carte and ...,0,1,3,2013,2013,drop,24,5.0,en,0,0.461132,10
1,qfvzHEL0gGxRdTo7NB2fnw,--g-a85VwrdZJNf0R95GcQ,D0YggUjK98hwIpiqXA9y_g,2013-11-18,5,I was in the Kabab House for the first time on...,0,1,3,2013,2013,drop,24,5.0,en,0,0.287281,44


41301

In [4]:
# Get the min and max date from the 'reviews' dataframe
min_date = pd.to_datetime(reviews["date"]).min()
second_oldest_date = pd.to_datetime(reviews["date"]).sort_values().iloc[1]
max_date = pd.to_datetime(reviews["date"]).max()
print(f"Minimum review date: {min_date}")
print(f"Maximum review date: {max_date}")

Minimum review date: 2010-01-03 00:00:00
Maximum review date: 2017-12-11 00:00:00


In [5]:
# create indices for training evaluation and calibration

from constants import CALIBRATION_INDICES, EVAL_INDICES, TRAIN_INDICES

# get original indices
if SET_ORIGINAL_INDICES:
    train_indices = TRAIN_INDICES
    calibration_indices = CALIBRATION_INDICES
    eval_indices = EVAL_INDICES

# get indices based on random seed
else:
    indices = np.random.permutation(len(business_covariates))
    indices_val_cal = np.random.permutation(
        np.arange(500, len(business_covariates))
    )  # range 500-921 (because of sorting)

    train_indices = indices[:500]  # take 500 random samples
    calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
    eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

In [6]:
assert (business_covariates.get("TRAIN")).sum() == 0, "training set already assigned!"

# set 'TRAIN' variable to 1 for train_indices, 0 otherwise
business_covariates.loc[train_indices, "TRAIN"] = 1

# sort business_covariates so that rows with Train==1 come first
business_covariates = business_covariates.sort_values(
    by="TRAIN", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :500

n_train = len(train_indices)  # number of training samples (500)

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 100 samples
  Eval: 321 samples


In [ ]:
# initialize list with data needed for stan

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")
print("Created required lists for MCMC sampling")

[0] - Conversion for business_id: QkG3KUXwqZBW18A9k1xqCA
[100] - Conversion for business_id: 9iy6scMRbs5ue41XRLGEkw
[200] - Conversion for business_id: nYpxA8exNB1VPuM7XYS7fg
[300] - Conversion for business_id: I7HGSg1OfAbO9X-mbDYdxg
[400] - Conversion for business_id: giCq1MmW-_S2tvNOAHvJcQ
[500] - Conversion for business_id: VMXqRzR3-SSVDVcP6VY4ag
[600] - Conversion for business_id: VaEqjAeKAm_iUnOpaktyMg
[700] - Conversion for business_id: Y_9f9tzVRaXjfT9Fa5_W9A
[800] - Conversion for business_id: 8XNpKETQcSLUPAROULyrOg
[900] - Conversion for business_id: KYkUtRB1QKrI6Twov98leA
Done ... validation checks passed!
Created required lists for MCMC sampling


In [8]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe
business_covariates["Age"] = age

# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)

# add log of age to dataframe for later analysis
business_covariates["logAge"] = np.log(business_covariates["Age"])

# only get relevant covariates for training
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,American,1,2.0,700.0,350.0,1248.0,2057
1,12,4.591304,Mexican,1,1.0,NaN,NaN,1710.0,1610
2,20,5.254499,Mexican,1,1.0,110.0,40.0,2007.0,389
3,17,10.490358,American,0,1.0,NaN,NaN,1557.0,363
4,20,10.778626,Mexican,0,2.0,40.0,15.0,1413.0,1834
...,...,...,...,...,...,...,...,...,...
916,5,3.540698,American,0,1.0,80.0,30.0,1892.0,688
917,57,5.005348,Fast Food,1,2.0,126.0,56.0,1521.0,2618
918,5,0.249894,Fast Food,1,1.0,130.0,50.0,1287.0,2353
919,2,0.524496,Asian,0,2.0,NaN,NaN,1253.0,694


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding already performed on dataframe!"

# Mark as categorial variable
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# One Hot Encoding
# also possible with OneHotEncoder and dropping first column (category_American), but wanting to retain the removed category (category_Other) in the original study
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

###### Create covariate matrix with relevant covariates

# First two numeric columns before category column
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# combine original order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# remove category_Other (8th column)
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (category_Other): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        # if not expected
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")

        # Still remove it
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (category_Other): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057


In [ ]:
imputer = SimpleImputer(strategy="median")  # impute column with median value
scaler = StandardScaler(with_std=False)  # only centering, no scaling!

X_train = relevant_covariates.iloc[:n_train].copy()

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition
Q, R = np.linalg.qr(X_train_preprocessed)

# scale the Q and R matrix appropriately
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

display(X_train)
display(pd.DataFrame(cov_mat_preprocessed))

,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057
1,12,4.591304,0,0,0,0,1,0,0,0,1,1.0,NaN,NaN,1710.0,1610
2,20,5.254499,0,0,0,0,1,0,0,0,1,1.0,110.0,40.0,2007.0,389
3,17,10.490358,1,0,0,0,0,0,0,0,0,1.0,NaN,NaN,1557.0,363
4,20,10.778626,0,0,0,0,1,0,0,0,0,2.0,40.0,15.0,1413.0,1834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,12,12.022599,1,0,0,0,0,0,0,0,0,2.0,NaN,NaN,NaN,177
496,11,3.152174,1,0,0,0,0,0,0,0,1,1.0,378.0,138.0,1469.0,1288
497,2,2.120332,0,0,0,0,1,0,0,0,0,2.0,297.0,94.0,1247.0,964
498,10,9.540000,1,0,0,0,0,0,0,0,0,2.0,NaN,NaN,1413.0,1400


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-1.226,-3.375542,0.744,-0.112,-0.076,-0.118,-0.178,-0.086,-0.044,-0.054,0.72,0.558,474.052,256.63,-241.134,731.298
1,-0.226,-1.111899,-0.256,-0.112,-0.076,-0.118,0.822,-0.086,-0.044,-0.054,0.72,-0.442,-69.948,-29.37,220.866,284.298
2,7.774,-0.448705,-0.256,-0.112,-0.076,-0.118,0.822,-0.086,-0.044,-0.054,0.72,-0.442,-115.948,-53.37,517.866,-936.702
3,4.774,4.787154,0.744,-0.112,-0.076,-0.118,-0.178,-0.086,-0.044,-0.054,-0.28,-0.442,-69.948,-29.37,67.866,-962.702
4,7.774,5.075422,-0.256,-0.112,-0.076,-0.118,0.822,-0.086,-0.044,-0.054,-0.28,0.558,-185.948,-78.37,-76.134,508.298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,-7.226,-2.162506,0.744,-0.112,-0.076,-0.118,-0.178,-0.086,-0.044,-0.054,-0.28,-0.442,-145.948,-63.37,402.866,-637.702
917,44.774,-0.697856,-0.256,-0.112,-0.076,0.882,-0.178,-0.086,-0.044,-0.054,0.72,0.558,-99.948,-37.37,31.866,1292.298
918,-7.226,-5.453310,-0.256,-0.112,-0.076,0.882,-0.178,-0.086,-0.044,-0.054,0.72,-0.442,-95.948,-43.37,-202.134,1027.298
919,-10.226,-5.178708,-0.256,0.888,-0.076,-0.118,-0.178,-0.086,-0.044,-0.054,-0.28,0.558,-69.948,-29.37,-236.134,-631.702


In [11]:
from helpers import comp_entropy

# aggregate review stats
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# Add variation coeffient to review_stats
review_stats["COV"] = np.sqrt(review_stats["VAR"]) / review_stats["MEAN"]

# select covariates, that are relevant for training the benchmark models
benchmark_covariates = business_covariates[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# Add Closed column (opposite from is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# merge covariates with aggregate review_stats
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# add logarithmic count
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical again
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# Convert Closed to categorical with proper labels (Closed = 1, Open = 0)
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")

benchmark_covariates

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Benchmark covariates prepared with shape: (921, 24)


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,COV,l_COUNT
0,QkG3KUXwqZBW18A9k1xqCA,11,2.327662,American,1,2.0,700.0,350.0,1248.0,14035.314000,...,2.648649,1.421063,37,0.432432,0.108108,0.081081,0.135135,0.243243,0.643046,3.610918
1,UJewG9UgVXQ7fyP4UHwnxw,12,4.591304,Mexican,1,1.0,NaN,NaN,1710.0,6876.845741,...,2.536585,1.522245,123,0.333333,0.260163,0.113821,0.121951,0.170732,0.584941,4.812184
2,ERM603jbIbNqX2c7Ww1Qiw,20,5.254499,Mexican,1,1.0,110.0,40.0,2007.0,18267.052850,...,3.171429,1.597666,35,0.171429,0.200000,0.171429,0.200000,0.257143,0.461936,3.555348
3,JLWd6yDyt9oEp70_4KqYeg,17,10.490358,American,0,1.0,NaN,NaN,1557.0,3273.245987,...,4.555556,0.832640,27,0.000000,0.037037,0.111111,0.111111,0.740741,0.185997,3.295837
4,prdA1r8XP03oD-PYvZJ5AA,20,10.778626,Mexican,0,2.0,40.0,15.0,1413.0,7630.840631,...,2.907692,1.582633,195,0.271795,0.153846,0.153846,0.235897,0.184615,0.512219,5.273000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,aLaIaJM9LsETSRPV1pllhg,5,3.540698,American,0,1.0,80.0,30.0,1892.0,7738.628975,...,3.263158,1.496514,38,0.105263,0.131579,0.289474,0.342105,0.131579,0.361085,3.637586
917,5-s0ESpKQeI50QIgcz8GGQ,57,5.005348,Fast Food,1,2.0,126.0,56.0,1521.0,443.034270,...,3.136364,1.555170,110,0.109091,0.209091,0.263636,0.272727,0.145455,0.389846,4.700480
918,pehIhPluVUAOuOdML6rFRA,5,0.249894,Fast Food,1,1.0,130.0,50.0,1287.0,5387.307080,...,1.700000,0.730588,20,0.750000,0.000000,0.150000,0.000000,0.100000,0.789200,2.995732
919,beqqBCKM_4aRPmRzCfk2xA,2,0.524496,Asian,0,2.0,NaN,NaN,1253.0,12310.711098,...,3.480000,1.217599,25,0.280000,0.000000,0.040000,0.320000,0.360000,0.477392,3.218876


In [12]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models (prepare_stan_data function)
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=train_indices,
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
output_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"
model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data_43.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Available
- Benchmark covariates shape: (921, 24)
- Columns: business_id, density, Checkin, category, chain...

